In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import argparse
from dataclasses import dataclass
from typing import Optional, Dict, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from fitparse import FitFile


# =========================
# ✅ CONFIG ATHLÈTE
# =========================
@dataclass
class AthleteConfig:
    sport: str                 # "bike" or "run"
    cp: float                  # CP (W) for bike, CS (m/s) for run
    lt1_hr: int                # bpm
    lt2_hr: int                # bpm

    # Zones mécaniques (en fraction de CP/CS)
    lt1_frac: float = 0.75
    lt2_frac: float = 0.90

    # Lissage HR (secondes)
    hr_smooth_window_s: int = 30

    # Split drift
    split_mode: str = "time"   # "time" recommended

    # Modèle mixte coefficients
    alpha_int: float = 0.50     # poids du temps HR dans LT1-LT2
    beta_dur: float = 0.08      # pénalité drift au-delà du seuil
    drift_threshold: float = 3.0 # % (Pa:HR drift abs) toléré

    # Pondération intensité (IF)
    if_bins: Tuple[Tuple[float, float], ...] = ((0.0, 0.75), (0.75, 0.85), (0.85, 0.92), (0.92, 9.0))
    if_factors: Tuple[float, ...] = (0.8, 1.0, 1.2, 1.4)

    # Nettoyage
    max_dt_s: float = 10.0      # on coupe les trous GPS >10s (pause)
    min_hr: int = 40
    max_hr: int = 230


# =========================
# 📥 LECTURE FIT
# =========================
def read_fit(filepath: str) -> pd.DataFrame:
    fitfile = FitFile(filepath)

    rows = []
    for msg in fitfile.get_messages("record"):
        v = msg.get_values()

        rows.append({
            "time": v.get("timestamp"),
            "power": v.get("power"),
            "hr": v.get("heart_rate"),
            "speed": v.get("speed"),  # m/s (souvent présent aussi à vélo)
            "cadence": v.get("cadence"),
            "distance": v.get("distance"),  # m
        })

    df = pd.DataFrame(rows)
    df = df.dropna(subset=["time"])
    df["time"] = pd.to_datetime(df["time"])
    df = df.sort_values("time").reset_index(drop=True)

    # dt (s)
    df["dt"] = df["time"].diff().dt.total_seconds()
    df.loc[df["dt"].isna(), "dt"] = 1.0

    return df


# =========================
# 🧼 NETTOYAGE & SMOOTH
# =========================
def preprocess(df: pd.DataFrame, cfg: AthleteConfig) -> pd.DataFrame:
    df = df.copy()

    # supprimer trous (pause / stop) : on met dt=0 pour ne pas compter
    df.loc[df["dt"] > cfg.max_dt_s, "dt"] = 0.0

    # HR cleaning
    if "hr" in df.columns:
        df.loc[(df["hr"] < cfg.min_hr) | (df["hr"] > cfg.max_hr), "hr"] = np.nan

    # power cleaning
    if "power" in df.columns:
        df.loc[df["power"] < 0, "power"] = np.nan

    # speed cleaning
    if "speed" in df.columns:
        df.loc[df["speed"] < 0, "speed"] = np.nan

    # drop rows where dt == 0 (pause) for analysis (but keep for plotting if needed)
    df_active = df[df["dt"] > 0].copy()
    df_active = df_active.dropna(subset=["hr"])  # HR nécessaire pour drift/zonage HR

    # HR smoothing by time-based rolling approximation:
    # assume 1Hz-ish; window ~ cfg.hr_smooth_window_s samples
    window = max(5, int(cfg.hr_smooth_window_s))
    df_active["hr_smooth"] = df_active["hr"].rolling(window, center=True, min_periods=int(window/3)).mean()
    df_active["hr_smooth"] = df_active["hr_smooth"].fillna(method="bfill").fillna(method="ffill")

    return df_active


# =========================
# 📊 ZONES & INDICATEURS
# =========================
def intensity_series(df: pd.DataFrame, cfg: AthleteConfig) -> pd.Series:
    if cfg.sport.lower() == "bike":
        if "power" not in df.columns:
            raise ValueError("Colonne 'power' absente. Impossible de calculer %CP.")
        return df["power"] / cfg.cp
    else:
        # run
        if "speed" not in df.columns:
            raise ValueError("Colonne 'speed' absente. Impossible de calculer %CS.")
        return df["speed"] / cfg.cp


def energy_kj(df: pd.DataFrame, cfg: AthleteConfig) -> Optional[float]:
    # Vélo: kJ mécaniques
    if cfg.sport.lower() == "bike" and "power" in df.columns:
        p = df["power"].fillna(0)
        kj = (p * df["dt"]).sum() / 1000.0
        return float(kj)
    # Course: on laisse None (tu pourras ajouter un modèle kcal plus tard)
    return None


def zone_time_by_threshold(series: pd.Series, dt: pd.Series, thr1: float, thr2: float) -> Tuple[float, float, float]:
    # returns seconds in <thr1, [thr1, thr2), >=thr2
    z1 = dt[series < thr1].sum()
    z2 = dt[(series >= thr1) & (series < thr2)].sum()
    z3 = dt[series >= thr2].sum()
    return float(z1), float(z2), float(z3)


def pick_intensity_factor(IF: float, cfg: AthleteConfig) -> float:
    for (lo, hi), f in zip(cfg.if_bins, cfg.if_factors):
        if lo <= IF < hi:
            return f
    return cfg.if_factors[-1]


def split_halves(df: pd.DataFrame, cfg: AthleteConfig) -> Tuple[pd.DataFrame, pd.DataFrame]:
    # split by time (recommended)
    if cfg.split_mode == "time":
        t = df["dt"].cumsum()
        half_time = t.iloc[-1] / 2.0
        first = df[t <= half_time]
        second = df[t > half_time]
    else:
        mid = len(df) // 2
        first = df.iloc[:mid]
        second = df.iloc[mid:]
    return first, second


def compute_metrics(df: pd.DataFrame, cfg: AthleteConfig) -> Dict[str, float]:
    df = df.copy()
    df["intensity"] = intensity_series(df, cfg)

    # totals
    total_time_s = float(df["dt"].sum())
    duration_h = total_time_s / 3600.0

    # energy
    e_kj = energy_kj(df, cfg)

    # IF (moyenne de %CP/CS sur données actives)
    IF = float(df["intensity"].mean())

    # zones mécaniques (%CP/CS)
    z1m, z2m, z3m = zone_time_by_threshold(df["intensity"], df["dt"], cfg.lt1_frac, cfg.lt2_frac)

    # zones HR (LT1_HR / LT2_HR)
    z1h, z2h, z3h = zone_time_by_threshold(df["hr_smooth"], df["dt"], cfg.lt1_hr, cfg.lt2_hr)

    # split halves
    first, second = split_halves(df, cfg)

    # Pa:HR
    if cfg.sport.lower() == "bike":
        p1 = float(first["power"].fillna(0).mean())
        p2 = float(second["power"].fillna(0).mean())
        hr1 = float(first["hr_smooth"].mean())
        hr2 = float(second["hr_smooth"].mean())
        pahr_1 = p1 / hr1 if hr1 > 0 else np.nan
        pahr_2 = p2 / hr2 if hr2 > 0 else np.nan
    else:
        s1 = float(first["speed"].fillna(0).mean())
        s2 = float(second["speed"].fillna(0).mean())
        hr1 = float(first["hr_smooth"].mean())
        hr2 = float(second["hr_smooth"].mean())
        pahr_1 = s1 / hr1 if hr1 > 0 else np.nan
        pahr_2 = s2 / hr2 if hr2 > 0 else np.nan

    drift_pahr_pct = float((pahr_2 / pahr_1 - 1) * 100) if (pahr_1 and not np.isnan(pahr_1)) else np.nan
    hr_drift_pct = float((hr2 / hr1 - 1) * 100) if hr1 > 0 else np.nan

    # =========================
    # 🧠 MODÈLE MIXTE
    # =========================
    # MEC = énergie × facteur(IF)
    intensity_factor = pick_intensity_factor(IF, cfg)

    if e_kj is None:
        MEC = np.nan
    else:
        MEC = e_kj * intensity_factor

    # INT = 1 + alpha * (%temps HR dans LT1–LT2)
    p_hr_lt = (z2h / total_time_s) if total_time_s > 0 else 0.0
    INT = 1.0 + cfg.alpha_int * p_hr_lt

    # DUR = 1 + beta * max(0, |drift Pa:HR| - seuil)
    drift_abs = abs(drift_pahr_pct) if not np.isnan(drift_pahr_pct) else np.nan
    if np.isnan(drift_abs):
        DUR = np.nan
    else:
        DUR = 1.0 + cfg.beta_dur * max(0.0, drift_abs - cfg.drift_threshold)

    # MLS final
    if np.isnan(MEC) or np.isnan(DUR):
        MLS = np.nan
    else:
        MLS = MEC * INT * DUR

    out = {
        "Duration_h": round(duration_h, 3),
        "IF_%CP_CS": round(IF, 3),

        "Energy_kJ": round(e_kj, 1) if e_kj is not None else np.nan,

        "Mech_%time_<LT1": round(100*z1m/total_time_s, 1) if total_time_s > 0 else np.nan,
        "Mech_%time_LT1_LT2": round(100*z2m/total_time_s, 1) if total_time_s > 0 else np.nan,
        "Mech_%time_>=LT2": round(100*z3m/total_time_s, 1) if total_time_s > 0 else np.nan,

        "HR_%time_<LT1": round(100*z1h/total_time_s, 1) if total_time_s > 0 else np.nan,
        "HR_%time_LT1_LT2": round(100*z2h/total_time_s, 1) if total_time_s > 0 else np.nan,
        "HR_%time_>=LT2": round(100*z3h/total_time_s, 1) if total_time_s > 0 else np.nan,

        "PaHR_1": round(pahr_1, 4) if not np.isnan(pahr_1) else np.nan,
        "PaHR_2": round(pahr_2, 4) if not np.isnan(pahr_2) else np.nan,
        "PaHR_drift_%": round(drift_pahr_pct, 2) if not np.isnan(drift_pahr_pct) else np.nan,
        "HR_drift_%": round(hr_drift_pct, 2) if not np.isnan(hr_drift_pct) else np.nan,

        "Intensity_factor": round(intensity_factor, 2),

        "MEC": round(MEC, 1) if not np.isnan(MEC) else np.nan,
        "INT": round(INT, 3),
        "DUR": round(DUR, 3) if not np.isnan(DUR) else np.nan,
        "MLS": round(MLS, 1) if not np.isnan(MLS) else np.nan,
    }

    return out


# =========================
# 📈 PLOTS (OPTION)
# =========================
def plot_session(df: pd.DataFrame, cfg: AthleteConfig, title: str, out_png: Optional[str] = None) -> None:
    df = df.copy()
    df["t_min"] = df["dt"].cumsum() / 60.0

    fig = plt.figure(figsize=(12, 6))
    ax1 = plt.gca()

    if cfg.sport.lower() == "bike":
        ax1.plot(df["t_min"], df["power"], linewidth=1)
        ax1.set_ylabel("Power (W)")
    else:
        ax1.plot(df["t_min"], df["speed"] * 3.6, linewidth=1)
        ax1.set_ylabel("Speed (km/h)")

    ax2 = ax1.twinx()
    ax2.plot(df["t_min"], df["hr_smooth"], linewidth=1)
    ax2.set_ylabel("HR (bpm)")

    ax1.set_xlabel("Time (min)")
    ax1.set_title(title)

    if out_png:
        plt.tight_layout()
        plt.savefig(out_png, dpi=180)
        plt.close(fig)
    else:
        plt.show()


def plot_pahr_halves(df: pd.DataFrame, cfg: AthleteConfig, title: str, out_png: Optional[str] = None) -> None:
    first, second = split_halves(df, cfg)

    if cfg.sport.lower() == "bike":
        p1 = float(first["power"].fillna(0).mean()); hr1 = float(first["hr_smooth"].mean())
        p2 = float(second["power"].fillna(0).mean()); hr2 = float(second["hr_smooth"].mean())
        pahr_1 = p1/hr1; pahr_2 = p2/hr2
        labels = ["1st half", "2nd half"]
        values = [pahr_1, pahr_2]
        ylab = "W / bpm"
    else:
        s1 = float(first["speed"].fillna(0).mean()); hr1 = float(first["hr_smooth"].mean())
        s2 = float(second["speed"].fillna(0).mean()); hr2 = float(second["hr_smooth"].mean())
        pahr_1 = s1/hr1; pahr_2 = s2/hr2
        labels = ["1st half", "2nd half"]
        values = [pahr_1, pahr_2]
        ylab = "(m/s) / bpm"

    fig = plt.figure(figsize=(6, 4))
    ax = plt.gca()
    ax.bar(labels, values)
    ax.set_ylabel(ylab)
    ax.set_title(title)
    plt.tight_layout()

    if out_png:
        plt.savefig(out_png, dpi=180)
        plt.close(fig)
    else:
        plt.show()


# =========================
# ▶️ MAIN
# =========================
def main():
    parser = argparse.ArgumentParser(description="Mixed load dashboard metrics from FIT")
    parser.add_argument("--fit", required=True, help="Path to .FIT activity file")
    parser.add_argument("--sport", required=True, choices=["bike", "run"], help="bike or run")
    parser.add_argument("--cp", type=float, required=True, help="CP (W) for bike, CS (m/s) for run")
    parser.add_argument("--lt1_hr", type=int, required=True, help="LT1 HR (bpm)")
    parser.add_argument("--lt2_hr", type=int, required=True, help="LT2 HR (bpm)")
    parser.add_argument("--out_csv", default="session_metrics.csv", help="Output CSV (appends a row)")
    parser.add_argument("--plots", action="store_true", help="Generate PNG plots next to CSV")
    args = parser.parse_args()

    cfg = AthleteConfig(
        sport=args.sport,
        cp=args.cp,
        lt1_hr=args.lt1_hr,
        lt2_hr=args.lt2_hr,
    )

    raw = read_fit(args.fit)
    df = preprocess(raw, cfg)
    metrics = compute_metrics(df, cfg)

    # print
    print("\n=== METRICS ===")
    for k, v in metrics.items():
        print(f"{k}: {v}")

    # append CSV
    row = pd.DataFrame([metrics])
    try:
        existing = pd.read_csv(args.out_csv)
        out = pd.concat([existing, row], ignore_index=True)
    except Exception:
        out = row

    out.to_csv(args.out_csv, index=False)
    print(f"\n✅ Saved/updated: {args.out_csv}")

    # plots
    if args.plots:
        base = args.out_csv.replace(".csv", "")
        plot_session(df, cfg, title="Session overview", out_png=f"{base}_overview.png")
        plot_pahr_halves(df, cfg, title="Pa:HR halves", out_png=f"{base}_pahr.png")
        print(f"✅ Plots saved: {base}_overview.png, {base}_pahr.png")


if __name__ == "__main__":
    main()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from fitparse import FitFile

# =========================
# 🔧 PARAMÈTRES ATHLÈTE (À MODIFIER)
# =========================
SPORT = "bike"
CP = 300          # Critical Power (W)
LT1_HR = 160      # bpm
LT2_HR = 172      # bpm

alpha_int = 0.50
beta_dur = 0.08
drift_threshold = 3.0

# =========================
# 📥 LECTURE FIT
# =========================
def read_fit(filepath):
    fitfile = FitFile(filepath)
    rows = []

    for msg in fitfile.get_messages("record"):
        v = msg.get_values()
        rows.append({
            "time": v.get("timestamp"),
            "power": v.get("power"),
            "hr": v.get("heart_rate"),
            "speed": v.get("speed")
        })

    df = pd.DataFrame(rows).dropna(subset=["time"])
    df["time"] = pd.to_datetime(df["time"])
    df["dt"] = df["time"].diff().dt.total_seconds().fillna(1)
    df.loc[df["dt"] > 10, "dt"] = 0  # pauses
    return df[df["dt"] > 0].copy()

# =========================
# 🧮 CALCULS
# =========================
def compute_metrics(df):
    df["hr_smooth"] = df["hr"].rolling(30, center=True).mean()
    df["hr_smooth"] = df["hr_smooth"].bfill().ffill()

    df["intensity"] = df["power"] / CP
    energy_kj = (df["power"] * df["dt"]).sum() / 1000

    total_time = df["dt"].sum()
    IF = df["intensity"].mean()

    # Zones HR
    z1 = df[df["hr_smooth"] < LT1_HR]["dt"].sum()
    z2 = df[(df["hr_smooth"] >= LT1_HR) & (df["hr_smooth"] < LT2_HR)]["dt"].sum()
    z3 = df[df["hr_smooth"] >= LT2_HR]["dt"].sum()

    # Split halves
    half_time = total_time / 2
    df["cum_t"] = df["dt"].cumsum()
    first = df[df["cum_t"] <= half_time]
    second = df[df["cum_t"] > half_time]

    PaHR_1 = first["power"].mean() / first["hr_smooth"].mean()
    PaHR_2 = second["power"].mean() / second["hr_smooth"].mean()

    drift_pahr = (PaHR_2 / PaHR_1 - 1) * 100
    hr_drift = (second["hr_smooth"].mean() /
                first["hr_smooth"].mean() - 1) * 100

    # Pondération IF
    if IF < 0.75:
        f_int = 0.8
    elif IF < 0.85:
        f_int = 1.0
    elif IF < 0.92:
        f_int = 1.2
    else:
        f_int = 1.4

    MEC = energy_kj * f_int
    INT = 1 + alpha_int * (z2 / total_time)
    DUR = 1 + beta_dur * max(0, abs(drift_pahr) - drift_threshold)

    MLS = MEC * INT * DUR

    return {
        "Energy_kJ": round(energy_kj, 1),
        "IF": round(IF, 3),
        "%HR_LT1": round(z1 / total_time * 100, 1),
        "%HR_LT1_LT2": round(z2 / total_time * 100, 1),
        "%HR_GT_LT2": round(z3 / total_time * 100, 1),
        "PaHR_drift_%": round(drift_pahr, 2),
        "HR_drift_%": round(hr_drift, 2),
        "MEC": round(MEC, 1),
        "INT": round(INT, 3),
        "DUR": round(DUR, 3),
        "MLS": round(MLS, 1)
    }, df

# =========================
# ▶️ EXECUTION
# =========================
fit_file = list(uploaded.keys())[0]
df = read_fit(fit_file)
metrics, df = compute_metrics(df)

pd.DataFrame([metrics])
